# 08 ? CPU: online versus batches materializados

## Registro do piloto

- Pergunta: quanto custa produzir batches online versus ler os mesmos batches de um cache local, e como workers CPU alteram esse custo?
- Seed raiz: 20261120; batches: 12; exemplos multivariados por batch: 8; canais: 6; contexto: 32; horizonte: 16.
- Gerador: configura??o sem operadores (baseline est?vel para validar o harness), noise std 0.02?0.05, independ?ncia entre canais.
- Hardware/software: preenchido pela c?lula de ambiente; sem Chronos e sem GPU.
- Uso: microbenchmark de infraestrutura; n?o representa o custo de equa??es complexas nem a execu??o do modelo.
- Cache: `outputs/` local, ignorado pelo Git; registro de execu??o inclu?do neste notebook.

A m?trica `next(loader)` ? espera observada no processo leitor. Neste CPU-only piloto, n?o ? m?trica de GPU stall.

## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/08_cpu_online_offline_scaling.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.


In [1]:
from pathlib import Path
import sys,time,platform,statistics
import numpy as np
import torch
ROOT=Path.cwd()
while not (ROOT/'src/tsfmxai').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/'src/tsfmxai').exists(): raise RuntimeError('repository root not found')
sys.path.insert(0,str(ROOT/'src'))
from tsfmxai.datasets.online import OnlineBatchPlan,CurriculumStage,make_dataloader
from tsfmxai.datasets.multiseries import TrainingFormat,TrajectoryConfig
from tsfmxai.generators.initialization import HistorySpec
from tsfmxai.generators.combinatorial_expression import CombinatorialTreeConfig
from tsfmxai.generators.multiseries import MultiseriesConfig,NoiseConfig
SEED=20261120; BATCHES=12; EXAMPLES=8; CHANNELS=6; CONTEXT=32; HORIZON=16
config=MultiseriesConfig(ast_config=CombinatorialTreeConfig(minimum_operators=0,maximum_operators=0,minimum_lag=1,maximum_lag=2,max_degree=2,require_lag=False),
 channel_count_probs={CHANNELS:1.0},dependency_probs={'independent':1.0},treatment_probs={'original':1.0},
 max_external_sources=1,allow_no_lags=True,noise=NoiseConfig(std_range=(0.02,0.05)))
plan=OnlineBatchPlan((CurriculumStage(0,config,EXAMPLES),),BATCHES,SEED,
 TrajectoryConfig(CONTEXT+HORIZON*3,HistorySpec('uniform','uniform')),
 TrainingFormat(CONTEXT,HORIZON,HORIZON,batch_size=EXAMPLES,dtype='float32'))
print({'python':platform.python_version(),'torch':torch.__version__,'device':'cpu','cuda':torch.cuda.is_available(),
 'platform':platform.platform(),'seed':SEED,'batches':BATCHES,'rows_per_batch':EXAMPLES*CHANNELS})

{'python': '3.11.9', 'torch': '2.12.0+cpu', 'device': 'cpu', 'cuda': False, 'platform': 'Windows-10-10.0.19045-SP0', 'seed': 20261120, 'batches': 12, 'rows_per_batch': 48}


## Materializar uma vez e checar igualdade

O cache preserva arrays sem compress?o para isolar custo de gera??o do custo de compress?o. Ap?s salvar, cada ID ? lido e comparado campo a campo ? gera??o determin?stica correspondente. O cache usa arquivos tempor?rios locais sob `outputs/` e n?o entra em Git.

In [2]:
CACHE=ROOT/'outputs'/'generator_model_evaluation'/'cpu_online_offline_cache'
CACHE.mkdir(parents=True,exist_ok=True)
fields=('context','future','channel_mask','equation_targets','equation_mask')
t0=time.perf_counter()
for batch_id in range(BATCHES):
 b=plan.generate(batch_id)
 np.savez(CACHE/f'batch-{batch_id:04d}.npz',**{k:getattr(b,k) for k in fields},noise_std=b.noise_targets['std'])
generation_seconds=time.perf_counter()-t0
t0=time.perf_counter()
for batch_id in range(BATCHES):
 b=plan.generate(batch_id)
 with np.load(CACHE/f'batch-{batch_id:04d}.npz') as f:
  for k in fields: assert np.array_equal(getattr(b,k),f[k]),(batch_id,k)
  assert np.array_equal(b.noise_targets['std'],f['noise_std'])
equality_seconds=time.perf_counter()-t0
cache_bytes=sum(p.stat().st_size for p in CACHE.glob('*.npz'))
print({'generation_to_cache_s':round(generation_seconds,3),'regenerate_and_compare_s':round(equality_seconds,3),'cache_MiB':round(cache_bytes/2**20,3)})

{'generation_to_cache_s': 0.153, 'regenerate_and_compare_s': 0.282, 'cache_MiB': 0.146}


## Tempo observado para obter os batches

Medimos o custo por batch em tr?s condi??es: gera??o sequencial (0 workers), gera??o com 1/2 workers, e leitura do cache via Dataset. As primeiras itera??es servem de warm-up. S?o tr?s repeti??es; reportamos mediana e p95. Ordem fixa neste piloto, portanto a medida ? descritiva, n?o uma compara??o estat?stica definitiva.

In [3]:
class CachedBatches(torch.utils.data.Dataset):
 def __len__(self): return BATCHES
 def __getitem__(self,i):
  with np.load(CACHE/f'batch-{int(i):04d}.npz') as f:
   return {k:torch.from_numpy(f[k].copy()) for k in (*fields,'noise_std')}
def measure(loader_factory,repeats=3,warmup=2):
 samples=[]
 for _ in range(repeats):
  loader=loader_factory(); it=iter(loader)
  for _ in range(warmup): next(it)
  for _ in range(BATCHES-warmup):
   t=time.perf_counter(); next(it); samples.append(time.perf_counter()-t)
  del it,loader
 return {'median_ms':float(np.median(samples)*1000),'p95_ms':float(np.quantile(samples,.95)*1000),
         'batches_per_s':float(1/np.mean(samples))}
results={}
for workers in (0,1,2):
 results[f'online_workers_{workers}']=measure(lambda w=workers:make_dataloader(plan,num_workers=w,prefetch_factor=1,pin_memory=False))
results['offline_cache']=measure(lambda:torch.utils.data.DataLoader(CachedBatches(),batch_size=None,num_workers=0))
results

{'online_workers_0': {'median_ms': 11.620550008956343,
  'p95_ms': 12.852795043727383,
  'batches_per_s': 86.14642077521245},
 'online_workers_1': {'median_ms': 16.30274998024106,
  'p95_ms': 27.79450998641549,
  'batches_per_s': 54.58742734071072},
 'online_workers_2': {'median_ms': 8.013950020540506,
  'p95_ms': 18.41346503351815,
  'batches_per_s': 113.83553267392195},
 'offline_cache': {'median_ms': 1.090400037355721,
  'p95_ms': 1.8521650577895343,
  'batches_per_s': 804.28738614971}}

In [4]:
import pandas as pd
summary=pd.DataFrame(results).T
summary

,median_ms,p95_ms,batches_per_s
online_workers_0,11.62055,12.852795,86.146421
online_workers_1,16.30275,27.794510,54.587427
online_workers_2,8.01395,18.413465,113.835533
offline_cache,1.09040,1.852165,804.287386


## Interpreta??o e limite

- `offline_cache` inclui leitura de NPZ e convers?o em tensores; n?o inclui gera??o nem grava??o do cache.
- `online_workers_*` inclui acesso ao gerador e entrega pelo DataLoader; sua medi??o n?o inclui Chronos, c?pia H2D ou passo de otimiza??o.
- O custo de prepara??o offline precisa ser amortizado: para N epochs, somar uma gera??o inicial ao tempo de cada ?poca offline.
- Workers podem perder em batches pequenos por custo de processo/serializa??o; avaliar a condi??o vencedora no tamanho real de treino.
- Sem CUDA e sem o modelo, nenhum n?mero deste notebook mede GPU stall. Para isso usar o protocolo do notebook 05 com `torch.profiler` e execu??o no mesmo hardware/modelo/batches.